# Diablo decision engine: fine-tune Laya typed-decisions (Kaggle, GPU T4 x2)

Trains `convaiinnovations/laya` / `typed-decisions` (Laya 0.3.20) on the Diablo decision dataset v1 with
upstream's RLCD recipe, Diablo's weighting, validation-based model selection and calibration on Diablo's
own calibration split.

**Settings:** Accelerator `GPU T4 x2`, Internet `On`.
**Input:** upload the repository folder `research/laya_diablo` as a Kaggle Dataset named `diablo-laya`
(it holds `contract.json`, `dataset/v1/`, `configs/`, `train/`).
**Output:** `/kaggle/working/laya-diablo-decision-v1.zip`: download it to the Mac.

The test splits are **not** evaluated here. Model selection uses validation only; the held-out test and
the private test are scored on the Mac with `research/laya_diablo/eval/eval_laya_diablo.py`,
against Diablo's rules on the same examples.

In [ ]:
!nvidia-smi
import torch, os
n = torch.cuda.device_count()
print("GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
assert n >= 1, "Select GPU T4 x2 under Notebook options -> Accelerator"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
!pip install -q "laya==0.3.20" "transformers>=4.48.0" safetensors huggingface_hub
import laya; print("laya", laya.__version__)
assert laya.__version__ == "0.3.20", "the manifest records the Laya version; pin it"


In [ ]:
import shutil, json, hashlib, glob
SRC = glob.glob("/kaggle/input/*/laya_diablo")[0] if glob.glob("/kaggle/input/*/laya_diablo") else glob.glob("/kaggle/input/diablo-laya*")[0]
WORK = "/kaggle/working/laya_diablo"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
manifest = json.load(open(f"{WORK}/dataset/v1/manifest.json"))
for split in ("train", "validation", "calibration"):
    digest = hashlib.sha256(open(f"{WORK}/dataset/v1/{split}.jsonl", "rb").read()).hexdigest()
    assert digest == manifest["split_hashes"][split], f"{split} does not match the dataset manifest"
print("dataset v1 verified:", {k: manifest["counts"][k] for k in ("train", "validation", "calibration")})


In [ ]:
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
root = snapshot_download("convaiinnovations/laya", allow_patterns=["typed-decisions/*"])
BASE = f"{root}/typed-decisions"
_fix_tokenizer_config(BASE)
print(json.load(open(f"{BASE}/rl_agent_config.json"))["model_name"])


In [ ]:
%cd /kaggle/working/laya_diablo/train
!python prepare_items.py --model-dir {BASE} --config ../configs/train_v1.json --out /kaggle/working/items


In [ ]:
!torchrun --standalone --nproc_per_node={max(1, torch.cuda.device_count())} train_diablo.py \
    --model-dir {BASE} --items /kaggle/working/items --out /kaggle/working/laya-diablo-decision-v1 \
    --config ../configs/train_v1.json --dataset-manifest ../dataset/v1/manifest.json


In [ ]:
out = "/kaggle/working/laya-diablo-decision-v1"
print(json.dumps(json.load(open(f"{out}/validation_by_epoch.json")), indent=1))
m = json.load(open(f"{out}/diablo_manifest.json"))
print("best epoch", m["training"]["best_epoch"], "temperatures", m["calibration"]["temperature"])
assert "temperature_by_options" not in json.load(open(f"{out}/rl_agent_config.json"))
shutil.make_archive(out, "zip", out)
print("download:", out + ".zip")


## On the Mac

    unzip laya-diablo-decision-v1.zip -d ~/diablo-models/laya-diablo-decision-v1
    python research/laya_diablo/eval/eval_laya_diablo.py score --engine laya \
        --checkpoint ~/diablo-models/laya-diablo-decision-v1 --device mps --out reports/laya_ft_test.json
    python research/laya_diablo/eval/eval_laya_diablo.py score --engine hybrid \
        --checkpoint ~/diablo-models/laya-diablo-decision-v1 --device mps --out reports/hybrid_ft_test.json
    python research/laya_diablo/eval/eval_laya_diablo.py longrun --checkpoint ... --device mps --out reports/longrun_ft.json
    python research/laya_diablo/eval/promotion.py --rules reports/rules_test.json \
        --laya reports/laya_ft_test.json --hybrid reports/hybrid_ft_test.json \
        --longrun reports/longrun_ft.json --fine-tuned true --calibrated true

Shadow it first: `DIABLO_DECISION_SHADOW=laya DIABLO_LAYA_CHECKPOINT=<path>`. Rules stay authoritative.